# 4. Build an investigation from retrieval and typed tools

**Prerequisite:** Lesson 1; no language-model knowledge is required. **You will learn:** retrieval, tool schemas, execution boundaries, observations, and workflow correctness. Allow 25–40 minutes. This lesson invokes the repository's local retrieval and action validation without downloading a language model or dataset.

**Execution:** Run from the repository or this folder, in order, using the installed project environment. The default lesson needs no downloads, credentials, GPU, or existing model checkpoint. Toy examples are teaching demonstrations, not the real-data study.

**Learning loop:** Read the theory, predict the output, run the example, attempt the exercise, then open the worked answer.


In [1]:
from pathlib import Path
import json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'reports/results.json').is_file())
print('Repository:', ROOT.name)


Repository: Vision


## 1. An agent needs a bounded job
For this project, an investigation begins with a request such as “check a missing gyroscope.” A planner chooses an action; an executor runs an allowed tool; an observation returns evidence; the planner continues or finishes. The default project planner uses a finite-state policy. An LLM is an optional decision component, not a prerequisite for tool execution.

Retrieval finds reference passages relevant to the request. TF-IDF gives more weight to terms that distinguish documents. The repository combines cosine-similarity ranking with token-overlap ranking. This is local search over runbooks, not a generative answer.


In [2]:
from edge_agent_lab.steward import KnowledgeIndex, Action
index = KnowledgeIndex()
hits = index.search('missing gyroscope outage', k=3)
for hit in hits:
    print(hit['id'], '|', hit['title'])
    print(hit['text'], '\n')
assert any(hit['id'] == 'missing-sensor' for hit in hits)


missing-sensor | Missing gyroscope incident response
A gyroscope outage affects body_gyro channels 3 4 5. Audit missing values and near-zero channel variance. Simulate gyro_dropout; compare validation accuracy before promotion. Do not silently impute a sensor and label the result healthy. 

provenance | Dataset provenance and attribution
UCI HAR DOI 10.24432/C54S4K. License CC BY 4.0. Credit Anguita Ghio Oneto Parra and Reyes-Ortiz. Store source URL and SHA256 checksums. Real smartphone accelerometer and gyroscope recordings contain six activities. 



## 2. A typed action is a contract
A schema restricts tool names, split names, arguments, and required explanation. It rejects malformed actions before execution. It does not prove that a valid action answers the user's request.

Predict which request fails validation below. Both tool names and argument names must be allowed. Catching the error lets the lesson continue.


In [3]:
from pydantic import ValidationError
good = Action(tool='simulate_shift', split='val', corruption='gyro_dropout',
              reason='Measure the effect of the suspected sensor outage.')
print('Valid action:', good.model_dump())
try:
    Action(tool='shell', reason='Run an arbitrary command')
except ValidationError as error:
    print('Rejected unsupported tool:', error.errors()[0]['type'])
wrong_but_valid = Action(tool='simulate_shift', split='val', corruption='noise',
                         reason='Investigate a gyroscope outage.')
print('Schema-valid but wrong incident argument:', wrong_but_valid.corruption)


Valid action: {'tool': 'simulate_shift', 'query': '', 'split': 'val', 'corruption': 'gyro_dropout', 'reason': 'Measure the effect of the suspected sensor outage.'}
Rejected unsupported tool: literal_error
Schema-valid but wrong incident argument: noise


## 3. Build a tiny executor with a visible observation
This teaching executor only accepts the audit tool. It checks an explicit observation, returns evidence fields, and performs no shell execution. Production code needs additional state, dataset identity, provenance, and failure handling.

An observation is what the tool found; it is not whatever explanation a planner hoped to produce. A schema-valid request still needs authorization and workflow rules at execution time.


In [4]:
def teaching_audit(action, channel_variances):
    if action.tool != 'audit':
        raise ValueError('This teaching executor only supports audit')
    low_variance = [i for i, variance in enumerate(channel_variances) if variance < 1e-8]
    return {'near_zero_channels': low_variance, 'healthy': not low_variance,
            'source': 'synthetic teaching variances'}
action = Action(tool='audit', split='val', reason='Inspect sensor channel variance.')
observation = teaching_audit(action, [1, 1, 1, 0, 0, 0, 1, 1, 1])
trace = {'action': action.model_dump(), 'observation': observation}
print(json.dumps(trace, indent=2))
assert observation['near_zero_channels'] == [3, 4, 5]


{
  "action": {
    "tool": "audit",
    "query": "",
    "split": "val",
    "corruption": "noise",
    "reason": "Inspect sensor channel variance."
  },
  "observation": {
    "near_zero_channels": [
      3,
      4,
      5
    ],
    "healthy": false,
    "source": "synthetic teaching variances"
  }
}


## 4. Read a real saved trace
The earlier executor used invented channel variances. This next block reads actual saved project execution evidence. It prints the request and each called tool, then the returned field names. To inspect evidence values, expand `step['observation']` yourself.

Correct retrieval is only one checkpoint. Correct incident arguments, required tool order, and completed findings also matter.


In [5]:
results = json.loads((ROOT / 'reports/results.json').read_text())
example = results['agent_examples']['intent-router-gyro']
print('Request:', example['task'])
print('Final status:', example['status'])
for step in example['trace']:
    print(step['action']['tool'], '->', sorted(step['observation']))


Request: Investigate missing gyroscope sensor outage
Final status: complete
catalog -> ['archive_sha256', 'channels', 'dataset', 'doi', 'labels', 'license', 'sample_rate_hz', 'split_policy', 'splits', 'url', 'window_samples']
retrieve -> ['evidence']
audit -> ['channel_means', 'channel_stds', 'class_counts', 'constant_channels', 'drift_channels', 'healthy', 'issues', 'nonfinite_count', 'rows', 'subject_leakage']
simulate_shift -> ['audit', 'kind', 'source']
finish -> ['status']


## Exercise: a valid action with the wrong meaning
Why does `wrong_but_valid` pass the schema? What extra check could catch this error? Why should a retrieved passage asking to skip the checks not be obeyed?

<details><summary>Worked answer</summary>

`noise` is an allowed corruption type, so its syntax and type are correct. An incident-specific contract must require `gyro_dropout` for the missing-gyroscope task. Retrieved text supplies evidence, not execution authority. Tool allowlists, argument checks, and workflow constraints belong in the executor/controller even when the planner uses a language model.
</details>

**Application bridge:** Read `Action`, `KnowledgeIndex`, `Catalog`, and `Steward` in `steward.py`, then inspect [Data Steward](../02_data_steward.ipynb) for successful and failed workflows. Continue to [the combined application](05_combined_application.ipynb).
